# Day 23 — AQE & Dynamic Partition Pruning

**What you will learn:**
- Adaptive Query Execution (AQE): the 3 runtime re-optimisations
- Dynamic Partition Pruning (DPP): how filter on dimension drives fact scan
- Coalescing post-shuffle partitions
- Skew join optimisation in AQE
- Key configs and how to verify AQE is working

**Exam relevance:** Optimisation (10%) — AQE features and DPP are explicitly tested.

In [ ]:
from pyspark.sql.functions import col, count, avg, sum, broadcast
import tempfile, os

LAKE = tempfile.mkdtemp(prefix="aqe_lab_")
print(f"Lake: {LAKE}")

# Verify AQE settings
for k in [
    "spark.sql.adaptive.enabled",
    "spark.sql.adaptive.coalescePartitions.enabled",
    "spark.sql.adaptive.skewJoin.enabled",
    "spark.sql.optimizer.dynamicPartitionPruning.enabled",
    "spark.sql.shuffle.partitions",
]:
    try:
        print(f"{k}: {spark.conf.get(k)}")
    except Exception:
        print(f"{k}: not set")

## 1. AQE — Why Static Planning Falls Short

Before Spark 3.0, the physical plan was fixed at compile time.
- Row counts were estimated from statistics (often wrong)
- A table estimated at 100MB could actually be 5MB after filtering
- A join estimated as two large sides could have one very small side

**AQE fixes this by re-optimising at runtime**, after shuffle data is materialised and real sizes are known.

## 2. The 3 AQE Optimisations

| Feature | What it does | Config |
|---|---|---|
| **Coalescing post-shuffle partitions** | Merges small adjacent partitions after shuffle | `coalescePartitions.enabled` (default true) |
| **Converting sort-merge to broadcast** | Switches to BHJ if one side turns out small | `autoBroadcastJoinThreshold` |
| **Skew join optimisation** | Splits skewed partitions into smaller sub-tasks | `skewJoin.enabled` (default true) |

## 3. Coalescing Post-Shuffle Partitions

In [ ]:
# With shuffle.partitions=200 and a small dataset,
# most partitions are empty → AQE merges them

spark.conf.set("spark.sql.shuffle.partitions", "200")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")

df = spark.range(1000).toDF("id") \
    .withColumn("bucket", (col("id") % 5).cast("string"))

grouped = df.groupBy("bucket").count()

# AQE wraps plan in AdaptiveSparkPlan
grouped.explain("formatted")

# After action: AQE decided actual partition count
result = grouped.collect()
print(f"Result rows: {len(result)}")

## 4. AQE Converting SortMerge to Broadcast Join

In [ ]:
# Disable broadcast at planning time — Spark plans SortMergeJoin
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

# But small_df is tiny — AQE detects this at runtime and switches to BHJ
large_df = spark.range(100000).withColumn("key", (col("id") % 10).cast("string"))
small_df = spark.createDataFrame([(str(i), f"name_{i}") for i in range(10)], ["key", "label"])

joined = large_df.join(small_df, on="key")

# Plan shows AdaptiveSparkPlan; at runtime may switch to BroadcastHashJoin
joined.explain("formatted")

# Re-enable for later cells
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10485760")

## 5. Skew Join Optimisation

In [ ]:
# Configure skew thresholds
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionFactor", "5")   # 5× median = skewed
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "256m")

print("Skew join enabled")
print("When a partition is 5× the median AND > 256MB, AQE splits it.")

## 6. Dynamic Partition Pruning (DPP)

DPP applies **partition pruning on a fact table** based on a runtime filter derived from a joined dimension table.

```
SELECT f.*, d.region
FROM   fact f
JOIN   dim  d ON f.date_id = d.date_id
WHERE  d.year = 2024
```

Without DPP: full fact table scan, then filter after join.
With DPP: Spark broadcasts the `date_id` set from `dim WHERE year=2024` and uses it to prune fact table partitions.

**Requirements for DPP:**
- Fact table must be **partitioned** on the join key
- Dimension side must be small enough to broadcast
- `spark.sql.optimizer.dynamicPartitionPruning.enabled = true` (default)

In [ ]:
# Create a partitioned fact table
fact_path = os.path.join(LAKE, "fact_sales")

from pyspark.sql.functions import lit

fact = spark.range(10000) \
    .withColumn("date_id", (col("id") % 365).cast("int")) \
    .withColumn("amount",  (col("id") * 1.5).cast("double"))

fact.write.mode("overwrite").partitionBy("date_id").parquet(fact_path)

# Small dimension
dim = spark.createDataFrame(
    [(i, 2024 if i < 30 else 2023) for i in range(365)],
    ["date_id", "year"]
)

fact_df = spark.read.parquet(fact_path)

# DPP: filter on dim.year propagates back to fact scan
result = fact_df.join(dim, on="date_id").filter(col("year") == 2024)

# Expect: PartitionFilters include dynamically generated filter from dim
result.explain("formatted")

print(f"Rows for year=2024: {result.count()}")

## 7. Key AQE/DPP Config Reference

| Config | Default | Effect |
|---|---|---|
| `spark.sql.adaptive.enabled` | true | Master AQE switch |
| `spark.sql.adaptive.coalescePartitions.enabled` | true | Merge empty post-shuffle partitions |
| `spark.sql.adaptive.coalescePartitions.minPartitionSize` | 1MB | Min size for merged partition |
| `spark.sql.adaptive.skewJoin.enabled` | true | Auto-split skewed partitions |
| `spark.sql.adaptive.skewJoin.skewedPartitionFactor` | 5 | Skew threshold multiplier |
| `spark.sql.optimizer.dynamicPartitionPruning.enabled` | true | Enable DPP |
| `spark.sql.autoBroadcastJoinThreshold` | 10MB | Broadcast if smaller |

---

## Day 23 Checklist

- [ ] Know the 3 AQE optimisations: coalesce, broadcast switch, skew split
- [ ] Know that AQE re-optimises using actual shuffle statistics
- [ ] Understand DPP: dimension filter propagated back to fact table partition scan
- [ ] Know DPP requirements: fact table partitioned on join key, dimension broadcastable
- [ ] Know key AQE configs and their defaults
- [ ] Verified `AdaptiveSparkPlan` wrapper in `explain()` output

**Next:** Day 24 — Delta Lake Fundamentals